# MLOps Platform - Test Notebook

Este cuaderno prueba el funcionamiento completo de la aplicación MLOps.

**Requisitos:** Los contenedores Docker deben estar corriendo (`docker compose up -d`).

**Nota:** Ejecutar las celdas en orden. Cada sección indica si pasó o falló el test.

In [ ]:
import requests
import json
import time
import hashlib
import hmac

BASE_URL = "http://localhost:8000"

def check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    msg = f"[{status}] {label}"
    if detail:
        msg += f" -> {detail}"
    print(msg)
    return condition

print("Setup listo. Base URL:", BASE_URL)

## 1. Health & Readiness Checks

In [ ]:
# Health check
r = requests.get(f"{BASE_URL}/health", timeout=10)
check("GET /health status", r.status_code == 200, f"status={r.status_code}")
check("GET /health body", r.json().get("status") == "ok", r.text)

In [ ]:
# Readiness check
r = requests.get(f"{BASE_URL}/ready", timeout=15)
data = r.json()
check("GET /ready status", r.status_code == 200, f"status={r.status_code}")
check("Redis reachable", data.get("redis") == "ok", f"redis={data.get('redis')}")
check("MLflow reachable", data.get("mlflow") == "ok", f"mlflow={data.get('mlflow')}")
print(f"  model_server: {data.get('model_server')}")
print(f"  overall: {data.get('status')}")

## 2. ROBLE API - Conexion Directa

Verifica que las credenciales ROBLE funcionan y las tablas existen.

In [ ]:
ROBLE_AUTH_URL = "https://roble-api.openlab.uninorte.edu.co/auth/mlops_platform_1d2a289c51"
ROBLE_DB_URL = "https://roble-api.openlab.uninorte.edu.co/database/mlops_platform_1d2a289c51"
ROBLE_EMAIL = "anzolas@uninorte.edu.co"
ROBLE_PASSWORD = "SeBaStIaN2004_"

# Login
r = requests.post(
    f"{ROBLE_AUTH_URL}/login",
    json={"email": ROBLE_EMAIL, "password": ROBLE_PASSWORD},
    timeout=15,
)
check("ROBLE login", r.status_code == 200, f"status={r.status_code}")
tokens = r.json()
access_token = tokens.get("accessToken", "")
check("accessToken received", len(access_token) > 0)

roble_headers = {"Authorization": f"Bearer {access_token}"}

In [ ]:
# Verify token
r = requests.get(f"{ROBLE_AUTH_URL}/verify-token", headers=roble_headers, timeout=10)
check("ROBLE verify-token", r.status_code == 200, f"status={r.status_code} body={r.text[:100]}")

In [ ]:
# Check tables exist
for table in ["repositories", "pipelines", "model_deployments"]:
    r = requests.get(
        f"{ROBLE_DB_URL}/read",
        params={"tableName": table},
        headers=roble_headers,
        timeout=15,
    )
    exists = r.status_code == 200
    count = len(r.json()) if exists and isinstance(r.json(), list) else "?"
    check(f"Table '{table}' exists", exists, f"status={r.status_code}, records={count}")

## 3. Repositories CRUD

In [ ]:
# List repos (GET /repos)
r = requests.get(f"{BASE_URL}/repos", timeout=15)
check("GET /repos status", r.status_code == 200, f"status={r.status_code}")
repos = r.json()
print(f"  Repos registrados: {len(repos)}")
for repo in repos:
    print(f"    - {repo.get('id')}: {repo.get('github_url')} (active={repo.get('is_active')})")

In [ ]:
# Register a test repo (POST /repos)
# Usa un repo publico de ejemplo. Si ya tienes un repo registrado, puedes saltar esta celda.
TEST_REPO_URL = "https://github.com/octocat/Hello-World"  # Repo publico de prueba

r = requests.post(
    f"{BASE_URL}/repos",
    json={
        "github_url": TEST_REPO_URL,
        "branch": "master",
        "notebook_path": "notebooks/train.ipynb",
    },
    timeout=30,
)
print(f"POST /repos -> status={r.status_code}")
print(f"  Response: {json.dumps(r.json(), indent=2)}")

# Guardar repo_id para las siguientes celdas
if r.status_code == 201:
    test_repo_id = r.json().get("repo_id")
    print(f"  repo_id guardado: {test_repo_id}")
else:
    test_repo_id = None
    print("  NOTA: El registro fallo. Esto puede ser normal si no tienes token de GitHub")
    print("  o si el repo ya esta registrado. Continua con los tests de lectura.")

In [ ]:
# Verify repo appears in list
r = requests.get(f"{BASE_URL}/repos", timeout=15)
repos_after = r.json()
print(f"  Total repos despues del registro: {len(repos_after)}")

if test_repo_id:
    found = any(repo.get("id") == test_repo_id for repo in repos_after)
    check("Repo aparece en listado", found, f"repo_id={test_repo_id}")

In [ ]:
# Delete test repo (cleanup)
if test_repo_id:
    r = requests.delete(f"{BASE_URL}/repos/{test_repo_id}", timeout=15)
    check("DELETE /repos/{id}", r.status_code == 200, f"status={r.status_code} body={r.text[:100]}")

    # Confirm deletion
    r = requests.get(f"{BASE_URL}/repos", timeout=15)
    found = any(repo.get("id") == test_repo_id for repo in r.json())
    check("Repo eliminado del listado", not found)
else:
    print("  SKIP: No hay repo de prueba para eliminar")

## 4. Pipelines

In [ ]:
# List pipelines (GET /pipelines)
r = requests.get(f"{BASE_URL}/pipelines", params={"page": 1, "size": 5}, timeout=15)
check("GET /pipelines status", r.status_code == 200, f"status={r.status_code}")
data = r.json()
print(f"  Total pipelines: {data.get('total', 0)}")
print(f"  Page: {data.get('page')}, Size: {data.get('size')}")

for p in data.get("items", []):
    print(f"    - {p.get('id')[:12]}... status={p.get('status')} repo={p.get('repo_id')} commit={p.get('commit_sha', 'N/A')[:8]}")

In [ ]:
# Get specific pipeline detail (if any exist)
if data.get("items"):
    pid = data["items"][0].get("id")
    r = requests.get(f"{BASE_URL}/pipelines/{pid}", timeout=15)
    check("GET /pipelines/{id}", r.status_code == 200, f"status={r.status_code}")
    detail = r.json()
    print(f"  Pipeline: {detail.get('id')}")
    print(f"  Status: {detail.get('status')}")
    print(f"  Phases: {json.dumps(detail.get('phases', []), indent=4)[:500]}")
    print(f"  Metrics: {detail.get('metrics')}")
else:
    print("  SKIP: No hay pipelines para consultar detalle")

In [ ]:
# Get pipeline logs (if any exist)
if data.get("items"):
    pid = data["items"][0].get("id")
    r = requests.get(f"{BASE_URL}/pipelines/{pid}/logs", timeout=15)
    check("GET /pipelines/{id}/logs", r.status_code in (200, 404), f"status={r.status_code}")
    if r.status_code == 200:
        logs_data = r.json()
        print(f"  Log entries: {len(logs_data.get('logs', []))}")
        for log in logs_data.get("logs", [])[:5]:
            print(f"    [{log.get('level')}] {log.get('phase')}: {log.get('message', '')[:80]}")
    else:
        print("  No logs found (normal if pipeline is old and logs expired from Redis)")
else:
    print("  SKIP: No hay pipelines")

In [ ]:
# Pipeline not found (404 test)
r = requests.get(f"{BASE_URL}/pipelines/nonexistent-id-12345", timeout=10)
check("GET /pipelines/nonexistent -> 404", r.status_code == 404, f"status={r.status_code}")

## 5. Models

In [ ]:
# List models (GET /models)
r = requests.get(f"{BASE_URL}/models", timeout=15)
check("GET /models status", r.status_code == 200, f"status={r.status_code}")
models = r.json()
print(f"  Active models: {len(models)}")
for m in models:
    print(f"    - {m.get('model_name')} v{m.get('version')} accuracy={m.get('accuracy')} endpoint={m.get('endpoint_url')}")

In [ ]:
# Predict (if a model is deployed)
if models:
    model_name = models[0].get("model_name")
    print(f"  Testing prediction with model: {model_name}")
    r = requests.post(
        f"{BASE_URL}/models/{model_name}/predict",
        json={"data": [[5.1, 3.5, 1.4, 0.2]]},
        timeout=30,
    )
    if r.status_code == 200:
        pred = r.json()
        check("POST /models/{name}/predict", True, f"prediction={pred.get('prediction')}")
    else:
        print(f"  Prediction returned status={r.status_code}: {r.text[:200]}")
        print("  (Esto puede ser normal si el model-server no tiene el modelo cargado)")
else:
    print("  SKIP: No hay modelos desplegados para probar prediccion")

## 6. Webhook Signature Verification

In [ ]:
WEBHOOK_SECRET = "47720cbd8a12fdbb05fcacd6075089a919f9a8ad835389f80cd39341c265436f"

# Test with INVALID signature -> should return 401
payload = json.dumps({"ref": "refs/heads/main"}).encode()
r = requests.post(
    f"{BASE_URL}/webhook/github",
    data=payload,
    headers={
        "Content-Type": "application/json",
        "X-Hub-Signature-256": "sha256=invalidsignature",
        "X-GitHub-Event": "push",
    },
    timeout=10,
)
check("Webhook invalid signature -> 401", r.status_code == 401, f"status={r.status_code}")

In [ ]:
# Test with VALID signature but non-push event -> should return 200 (ignored)
payload = json.dumps({"action": "opened"}).encode()
sig = "sha256=" + hmac.new(WEBHOOK_SECRET.encode(), payload, hashlib.sha256).hexdigest()
r = requests.post(
    f"{BASE_URL}/webhook/github",
    data=payload,
    headers={
        "Content-Type": "application/json",
        "X-Hub-Signature-256": sig,
        "X-GitHub-Event": "ping",
    },
    timeout=10,
)
check("Webhook ping event -> 200", r.status_code == 200, f"status={r.status_code} body={r.text[:100]}")

In [ ]:
# Test with VALID signature, push event, but no matching repo -> 200 or 404
payload = json.dumps({
    "ref": "refs/heads/main",
    "repository": {
        "html_url": "https://github.com/nonexistent/repo",
        "clone_url": "https://github.com/nonexistent/repo.git",
    },
    "head_commit": {"id": "abc123"},
    "commits": [{"modified": ["train.ipynb"], "added": [], "removed": []}],
}).encode()
sig = "sha256=" + hmac.new(WEBHOOK_SECRET.encode(), payload, hashlib.sha256).hexdigest()
r = requests.post(
    f"{BASE_URL}/webhook/github",
    data=payload,
    headers={
        "Content-Type": "application/json",
        "X-Hub-Signature-256": sig,
        "X-GitHub-Event": "push",
    },
    timeout=10,
)
check("Webhook push with no matching repo", r.status_code in (200, 404), f"status={r.status_code} body={r.text[:150]}")

## 7. ROBLE CRUD Directo (Insert / Read / Update / Delete)

In [ ]:
# Re-login to ensure fresh token
r = requests.post(
    f"{ROBLE_AUTH_URL}/login",
    json={"email": ROBLE_EMAIL, "password": ROBLE_PASSWORD},
    timeout=15,
)
access_token = r.json()["accessToken"]
roble_headers = {"Authorization": f"Bearer {access_token}"}
check("ROBLE re-login", r.status_code == 200)

In [ ]:
# INSERT a test record into repositories
test_record = {
    "github_url": "https://github.com/test/notebook-test-crud",
    "github_token_masked": "****test",
    "branch": "main",
    "notebook_path": "notebooks/test.ipynb",
    "webhook_id": 0,
    "webhook_url": "https://example.com/webhook",
    "created_at": "2026-03-20T00:00:00",
    "is_active": True,
}

r = requests.post(
    f"{ROBLE_DB_URL}/insert",
    json={"tableName": "repositories", "records": [test_record]},
    headers=roble_headers,
    timeout=15,
)
check("ROBLE INSERT", r.status_code == 200 or r.status_code == 201, f"status={r.status_code}")
insert_data = r.json()
print(f"  Response: {json.dumps(insert_data, indent=2)[:300]}")

# Extract _id of inserted record
inserted_id = None
if isinstance(insert_data, dict) and "inserted" in insert_data:
    ins = insert_data["inserted"]
    if ins and isinstance(ins, list):
        inserted_id = ins[0].get("_id")
elif isinstance(insert_data, list) and insert_data:
    inserted_id = insert_data[0].get("_id")
print(f"  Inserted _id: {inserted_id}")

In [ ]:
# READ the record back
if inserted_id:
    r = requests.get(
        f"{ROBLE_DB_URL}/read",
        params={"tableName": "repositories", "_id": inserted_id},
        headers=roble_headers,
        timeout=15,
    )
    check("ROBLE READ", r.status_code == 200, f"status={r.status_code}")
    records = r.json() if isinstance(r.json(), list) else []
    check("Record found", len(records) > 0, f"count={len(records)}")
    if records:
        print(f"  Record: {json.dumps(records[0], indent=2)[:300]}")
else:
    print("  SKIP: No inserted_id")

In [ ]:
# UPDATE the record
if inserted_id:
    r = requests.put(
        f"{ROBLE_DB_URL}/update",
        json={
            "tableName": "repositories",
            "idColumn": "_id",
            "idValue": inserted_id,
            "updates": {"branch": "develop", "is_active": False},
        },
        headers=roble_headers,
        timeout=15,
    )
    check("ROBLE UPDATE", r.status_code == 200, f"status={r.status_code} body={r.text[:150]}")

    # Verify update
    r = requests.get(
        f"{ROBLE_DB_URL}/read",
        params={"tableName": "repositories", "_id": inserted_id},
        headers=roble_headers,
        timeout=15,
    )
    if r.status_code == 200 and isinstance(r.json(), list) and r.json():
        rec = r.json()[0]
        check("Branch updated", rec.get("branch") == "develop", f"branch={rec.get('branch')}")
        check("is_active updated", rec.get("is_active") == False, f"is_active={rec.get('is_active')}")
else:
    print("  SKIP: No inserted_id")

In [ ]:
# DELETE the test record (cleanup)
if inserted_id:
    r = requests.delete(
        f"{ROBLE_DB_URL}/delete",
        json={
            "tableName": "repositories",
            "idColumn": "_id",
            "idValue": inserted_id,
        },
        headers=roble_headers,
        timeout=15,
    )
    check("ROBLE DELETE", r.status_code == 200, f"status={r.status_code} body={r.text[:150]}")

    # Verify deletion
    r = requests.get(
        f"{ROBLE_DB_URL}/read",
        params={"tableName": "repositories", "_id": inserted_id},
        headers=roble_headers,
        timeout=15,
    )
    records = r.json() if isinstance(r.json(), list) else r.json()
    is_empty = (isinstance(records, list) and len(records) == 0) or records == []
    check("Record deleted", is_empty, f"records={records}")
else:
    print("  SKIP: No inserted_id")

## 8. Frontend

In [ ]:
# Check frontend is serving
r = requests.get("http://localhost:3000", timeout=10)
check("Frontend serves HTML", r.status_code == 200, f"status={r.status_code}")
check("Contains React root", "root" in r.text or "app" in r.text.lower(), f"content_length={len(r.text)}")

## 9. MLflow

In [ ]:
# Check MLflow tracking server
r = requests.get("http://localhost:5000/api/2.0/mlflow/experiments/search", timeout=10)
check("MLflow API reachable", r.status_code == 200, f"status={r.status_code}")
experiments = r.json().get("experiments", [])
print(f"  Experiments: {len(experiments)}")
for exp in experiments[:5]:
    print(f"    - {exp.get('name')} (id={exp.get('experiment_id')})")

## 10. Docker Services Status

In [ ]:
import subprocess

result = subprocess.run(
    ["docker", "compose", "ps", "--format", "json"],
    capture_output=True, text=True, cwd=r"C:\Users\user10\Downloads\KUBEFLOW_IA_TEST"
)

if result.returncode == 0:
    lines = result.stdout.strip().split("\n")
    print(f"{'Service':<20} {'State':<12} {'Status':<25} {'Ports'}")
    print("-" * 80)
    for line in lines:
        try:
            svc = json.loads(line)
            name = svc.get("Service", svc.get("Name", "?"))
            state = svc.get("State", "?")
            status = svc.get("Status", "?")
            ports = svc.get("Publishers", svc.get("Ports", ""))
            if isinstance(ports, list):
                ports = ", ".join(f"{p.get('PublishedPort', '?')}->{p.get('TargetPort', '?')}" for p in ports if p.get('PublishedPort'))
            print(f"{name:<20} {state:<12} {status:<25} {ports}")
        except json.JSONDecodeError:
            print(line)
else:
    print("Error running docker compose ps:", result.stderr[:200])

## Resumen

Si todos los checks muestran `[PASS]`, la aplicacion esta funcionando correctamente:

- **Health/Ready**: Backend responde y conecta a Redis/MLflow
- **ROBLE**: Autenticacion y tablas funcionan
- **Repos CRUD**: Crear, listar y eliminar repositorios via API
- **Pipelines**: Listado y consulta de pipelines
- **Models**: Listado de modelos desplegados
- **Webhook**: Verificacion de firma HMAC funciona
- **ROBLE CRUD directo**: Insert/Read/Update/Delete en tablas
- **Frontend**: Sirve la SPA correctamente
- **MLflow**: Tracking server accesible
- **Docker**: Todos los servicios corriendo